# 7. JEV triplet review

Review the Ukrainian dataset and compare model and human decisions on one
question: **Does this triplet have a problem?** Use yes/no judgments only.
Individual human disagreements remain in the analysis. Issue types are not
agreement targets.

Install `python -m pip install -e '.[notebooks,review]'` from the WarBias directory.
Default execution replays the saved decisions and recomputes agreement offline.
Set the model-run flags and provide `JEV_API_KEY` only for new service requests.

In [1]:
import json
import os
from pathlib import Path

from pandas import DataFrame, set_option

from fairforget.dataset_workflow import create_workspace, find_project
from fairforget.io import read_rows, write_json
from fairforget.notebook_display import display
from fairforget.warbias.annotation_agreement import agreement_report, binary_agreement
from fairforget.warbias.annotation_records import load_review_tables, prepare_review_inputs
from fairforget.warbias.jev_review import JevClient, review_request, run_review, validate_response

project = find_project()
work = create_workspace("07_jev_review")
config = json.loads((project / "configs/review/jev.json").read_text())
protocol = json.loads((project / "protocol/binary_annotation_agreement.json").read_text())
prompt = (project / config["prompt"]).read_text()
issues = list(config["issues"])
metadata, human_reviews, saved_model_reviews, recorded_run, examples = load_review_tables(
    project, issues
)
set_option("display.max_colwidth", None)
set_option("display.max_rows", 60)

## Dataset and recorded run

The current dataset includes veterans and IDPs. It retains 1,059 original
triplets and adds 10,938 synthetic triplets. The saved run reviewed all 11,997
items with JEV. The tables below show model flags; a flag is a model judgment about
the triplet. No flagged item is silently removed.

The repository includes compact, de-identified tables for the full human overlap.
The complete run summary covers all items. Raw service responses and the full
synthetic corpus stay in the run directory, outside the code repository.

In [2]:
display(
    {
        "status": recorded_run["status"],
        "triplets": recorded_run["annotation_count"],
        "model": recorded_run["returned_models"],
        "rubric": recorded_run["rubric_id"],
        "request_bindings_verified": recorded_run["request_bindings_verified"],
    }
)
display(
    DataFrame(recorded_run["counts"])[["cohort", "target_group", "triplets", "flagged_triplets"]]
)
display(recorded_run["human_import"])

{'status': 'complete',
 'triplets': 11997,
 'model': {'jev-1.14.0': 11997},
 'rubric': 'warbias-triplet-review-v1',
 'request_bindings_verified': 11997}

,cohort,target_group,triplets,flagged_triplets
0,human,veteran,625,201
1,human,idp,434,112
2,synthetic,veteran,7118,555
3,synthetic,idp,3820,407


{'binary_judgments': 4129,
 'excluded_rows': {'repeat_person_item': 11, 'undone': 31},
 'issue_sets_known': 3830,
 'items': 1059,
 'review_rows': 4130,
 'reviewer_ids': 53}

## The triplet prompt and choices

S must express a stereotype. C must contradict that stereotype. U must be neutral
about the same group. Check natural Ukrainian and comparable sentence structure.
A style preference alone is not an error. The intended stereotype in S is not
itself a defect.

Each issue has present/absent choices in the same request. No problem means that
all seven flags are absent. The other flag covers a substantive issue outside
the six named categories. JEV does not rewrite sentences.

In [3]:
display(
    DataFrame(
        [
            {"issue": name, "human_label": value["label"], "question": value["instructions"]}
            for name, value in config["issues"].items()
        ]
    )
)
print(prompt)
example = examples[0]
request = review_request(example["item"], config, prompt)
validate_response(example["response"], config)
display(request["state"])
display(
    DataFrame(
        [
            {
                "issue": name,
                "choice": value["choice"],
                "confidence": value["confidence"],
                "problem_probability": value["probabilities"]["present"],
            }
            for name, value in example["response"]["answers"].items()
        ]
    )
)

,issue,human_label,question
0,grammar_or_spelling,Граматика чи правопис,"Does S, C or U contain an actual Ukrainian grammar or spelling error?"
1,unnatural,Звучить неприродно,"Does S, C or U contain unnatural Ukrainian, a calque or awkward wording that is a real language defect rather than a style preference?"
2,counter_not_contradicting,Не заперечує стереотип,Does C fail to contradict the particular stereotype expressed in S?
3,unrelated_implies_stereotype,Натякає на стереотип,Does U express or imply the stereotype instead of being neutral?
4,group_or_meaning,Інша група або зміст,"Does a sentence change the target group incorrectly or lose the intended stereotype meaning? C must address S, while different neutral content in U is allowed."
5,non_parallel,Трійка непаралельна,Do sentence structures or target descriptions differ so much that the three sentences are not parallel? Identical wording is not required.
6,other,Інше,"Is there another substantive failure of the review rules not covered by the six named issues, such as S not expressing a recognizable group stereotype?"


Review one Ukrainian S/C/U triplet using the human annotation rules.
S must express a recognizable stereotype about the target group. Its intended
stereotype is not itself an error. C must contradict that same stereotype.
U must be neutral about the same group and must not imply the stereotype.
All three sentences must use correct, natural Ukrainian. Keep the target group
and comparable sentence structure. A style preference alone is not an error.
The category and definition provide context. Different neutral content in U
is allowed. Judge the text under these rules, not the truth of the stereotype.
Each question checks one issue in the triplet. Multiple issues can be present.
Use the other-issue flag only for a substantive rule failure not covered by the
six named issues. Do not rewrite the text or add new review requirements.



{'review_rules': 'Review one Ukrainian S/C/U triplet using the human annotation rules.\nS must express a recognizable stereotype about the target group. Its intended\nstereotype is not itself an error. C must contradict that same stereotype.\nU must be neutral about the same group and must not imply the stereotype.\nAll three sentences must use correct, natural Ukrainian. Keep the target group\nand comparable sentence structure. A style preference alone is not an error.\nThe category and definition provide context. Different neutral content in U\nis allowed. Judge the text under these rules, not the truth of the stereotype.\nEach question checks one issue in the triplet. Multiple issues can be present.\nUse the other-issue flag only for a substantive rule failure not covered by the\nsix named issues. Do not rewrite the text or add new review requirements.\n',
 'category': 'Агресія, насильство, злочинність і залежності',
 'definition': 'A predisposition to drug dependence attributed to 

,issue,choice,confidence,problem_probability
0,counter_not_contradicting,absent,0.80,0.10
1,grammar_or_spelling,absent,0.92,0.04
2,group_or_meaning,absent,0.98,0.01
3,non_parallel,absent,0.26,0.37
4,other,absent,0.96,0.02
5,unnatural,absent,0.70,0.15
6,unrelated_implies_stereotype,absent,1.00,0.00


## Human review records

Use the original independent judgments before adjudication. In the app, yes
means no problem; no or an edit means a problem. Ignore undone rows. Retain the
first active judgment for a person and item. Imported round-one judgments may
have missing sentence reviews; partial acceptance is not a complete no-problem
judgment. Missing reasons stay unknown. Generic redo labels do not become new
issue categories. Current free-text reasons use other without reinterpreting their intent.
Legacy comments without an explicit category stay untyped. A partially typed
problem set remains unknown.

Reviewer IDs are pseudonyms. The public tables contain no names, private comments
or credential values. Text and context hashes bind human and model decisions to
the same version. The human text matches the pinned dataset reference.

In [4]:
display(
    DataFrame(
        [
            {
                "source": source,
                "review_rows": sum(row["source"] == source for row in human_reviews),
                "binary_judgments": sum(
                    row["source"] == source and row["has_problem"] is not None
                    for row in human_reviews
                ),
            }
            for source in sorted({row["source"] for row in human_reviews})
        ]
    )
)
display(
    {
        "human_triplets": len(metadata),
        "reviewer_ids": len({row["reviewer"] for row in human_reviews}),
        "review_rows": len(human_reviews),
        "known_binary_judgments": sum(row["has_problem"] is not None for row in human_reviews),
    }
)

,source,review_rows,binary_judgments
0,current,1508,1508
1,round1,2622,2621


{'human_triplets': 1059,
 'reviewer_ids': 53,
 'review_rows': 4130,
 'known_binary_judgments': 4129}

## Run or resume JEV

Set `SYNTHETIC_UK_PATH` to the Ukrainian triplets from notebook 6. The original
1,059 texts are loaded from the pinned reference, or from `HUMAN_UK_PATH` when
provided. The builder checks exact human text versions and the selected synthetic
IDs and profiles before making requests.

Each response is cached against its exact request. Keep `REVIEW_DIRECTORY` to
resume the same run. Use a new directory when text, context, rubric or model
changes. Temporary service errors are retried. A failed request never becomes
a no-problem label. The returned model must match the configured model ID.

In [5]:
RUN_JEV = False
SYNTHETIC_UK_PATH = Path("runs/coverage_generation/triplets.jsonl")
HUMAN_UK_PATH = None
REVIEW_DIRECTORY = Path("runs/jev_review")

model_reviews = saved_model_reviews
execution = {"status": "saved_annotation_replay", "new_service_requests": False}
if RUN_JEV:
    client = JevClient(
        os.environ.get("JEV_API_KEY"),
        timeout=config["transport"]["timeout_seconds"],
        attempts=config["transport"]["attempts"],
    )
    if HUMAN_UK_PATH is None:
        from fairforget.translation.reference import load_reference

        reference = json.loads((project / "configs/data/warbias-reference.json").read_text())
        human_rows, _, _ = load_reference(reference, cache_dir=work / "reference-cache")
    else:
        human_rows = read_rows(HUMAN_UK_PATH)
    items = prepare_review_inputs(project, human_rows, read_rows(SYNTHETIC_UK_PATH), metadata)
    execution = run_review(
        items,
        config,
        prompt,
        REVIEW_DIRECTORY,
        client=client,
        execute=True,
    )
    annotations = read_rows(REVIEW_DIRECTORY / "annotations.jsonl")
    model_reviews = [row for row in annotations if row["cohort"] == "human"]
display(execution)

{'status': 'saved_annotation_replay', 'new_service_requests': False}

## Agreement on the complete human cohort

The primary coefficient is the unweighted mean of pairwise Gwet AC1 values
between JEV and each human reviewer. Report each reviewer and their overlap
count. Measure human-human agreement separately; combining JEV and all humans
in one coefficient would not isolate model agreement.

Also report observed agreement with equal weight per triplet, the binary cross
counts, and class-specific agreement. Krippendorff nominal alpha describes the
human-only matrix with missing reviews. Human judgments are comparison partners,
not assumed error-free gold labels.

Confidence intervals use 2,000 bootstrap samples of whole semantic case families,
with seed 0. All variants and associated ratings stay together. The intervals
reflect case variation, not sampling new reviewers. Pair means describe different
review overlaps and do not establish human-model equivalence.

In [6]:
report = agreement_report(
    model_reviews,
    human_reviews,
    None,
    resamples=protocol["bootstrap"]["resamples"],
    seed=protocol["bootstrap"]["seed"],
)
overall = report["overall"]
summary = []
for label, value, interval in [
    (
        "JEV-human mean pair AC1",
        overall["jev_human"]["mean_pair_ac1"],
        overall["jev_human"]["mean_pair_ac1_ci95"],
    ),
    (
        "JEV-human item-weighted observed agreement",
        overall["jev_human"]["item_weighted_observed_agreement"]["value"],
        overall["jev_human"]["item_weighted_observed_agreement"]["ci95"],
    ),
    (
        "Human-human mean pair AC1",
        overall["human_human"]["mean_pair_ac1"],
        overall["human_human"]["mean_pair_ac1_ci95"],
    ),
    (
        "Human-only nominal alpha",
        overall["human_human"]["nominal_alpha"],
        overall["human_human"]["nominal_alpha_ci95"],
    ),
]:
    summary.append(
        {
            "measure": label,
            "value": value,
            "ci95_low": interval["low"],
            "ci95_high": interval["high"],
        }
    )
display(DataFrame(summary))
display(overall["coverage"])

,measure,value,ci95_low,ci95_high
0,JEV-human mean pair AC1,0.339081,0.261639,0.435816
1,JEV-human item-weighted observed agreement,0.667060,0.613958,0.715877
2,Human-human mean pair AC1,0.369550,0.336612,0.438105
3,Human-only nominal alpha,0.109501,0.075076,0.142122


{'human_items': 1059,
 'human_review_rows': 4130,
 'reviewer_ids': 53,
 'case_families': 127,
 'paired_binary_judgments': 4129,
 'missing_prediction_ids': [],
 'unknown_issue_sets': None,
 'both_empty_issue_pairs': None}

In [7]:
display(
    DataFrame(
        [
            {
                "target_group": group,
                "triplets": section["coverage"]["human_items"],
                "paired_judgments": section["coverage"]["paired_binary_judgments"],
                "mean_pair_ac1": section["jev_human"]["mean_pair_ac1"],
                "ci95_low": section["jev_human"]["mean_pair_ac1_ci95"]["low"],
                "ci95_high": section["jev_human"]["mean_pair_ac1_ci95"]["high"],
            }
            for group, section in report["by_group"].items()
        ]
    )
)
pair_table = DataFrame(
    [
        {key: value for key, value in row.items() if key != "ac1_ci95"}
        | {"ci95_low": row["ac1_ci95"]["low"], "ci95_high": row["ac1_ci95"]["high"]}
        for row in overall["jev_human"]["pairs"]
    ]
)
display(pair_table)
pair_table.to_csv(work / "jev-human-agreement.csv", index=False)

,target_group,triplets,paired_judgments,mean_pair_ac1,ci95_low,ci95_high
0,idp,434,1664,0.422379,0.287048,0.562116
1,veteran,625,2465,0.316479,0.211653,0.427985


,reviewer,n,ac1,observed_agreement,positive_agreement,negative_agreement,left_problem_rate,right_problem_rate,both_no_problem,right_only_problem,left_only_problem,both_problem,ci95_low,ci95_high
0,human_022f043b3d5f4dcd,10,0.009901,0.500000,0.444444,0.545455,0.400000,0.500000,3,3,2,2,-0.621622,0.675676
1,human_0411b09bdde39866,14,0.696751,0.785714,0.400000,0.869565,0.285714,0.071429,10,0,3,1,0.121951,1.000000
2,human_0b2051da99f00b6a,4,1.000000,1.000000,1.000000,1.000000,0.250000,0.250000,3,0,0,1,1.000000,1.000000
3,human_198da69adaedb1c5,170,0.475578,0.647059,0.142857,0.777778,0.300000,0.111765,105,14,46,5,0.311814,0.620691
4,human_203ab6deb902a230,3,-0.200000,0.333333,0.000000,0.500000,0.000000,0.666667,1,2,0,0,-1.000000,1.000000
5,human_285423550e20ac7f,9,0.538462,0.666667,0.000000,0.800000,0.333333,0.000000,6,0,3,0,-0.201622,1.000000
6,human_297e91ab1201383f,2,-1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0,2,0,0,-1.000000,-1.000000
7,human_2bfdec2ed3b0f4b7,26,0.692308,0.807692,0.615385,0.871795,0.230769,0.269231,17,3,2,4,0.397240,0.931710
8,human_2ddeecfdd2984c64,10,0.655172,0.800000,0.666667,0.857143,0.200000,0.400000,6,2,0,2,0.172414,1.000000
9,human_3425caf3dad4aa90,170,0.525613,0.670588,0.125000,0.797101,0.305882,0.070588,110,8,48,4,0.371247,0.656781


In [8]:
counts = [
    sum(row[key] for row in overall["jev_human"]["pairs"])
    for key in ["both_no_problem", "right_only_problem", "left_only_problem", "both_problem"]
]
display(
    DataFrame(
        [[counts[0], counts[1]], [counts[2], counts[3]]],
        index=["JEV: no problem", "JEV: problem"],
        columns=["Human: no problem", "Human: problem"],
    ).reset_index(names="model judgment")
)
class_agreement = binary_agreement(counts)
display(
    {
        "agreement_on_problems": class_agreement["positive_agreement"],
        "agreement_on_no_problem": class_agreement["negative_agreement"],
        "jev_problem_rate_on_pairs": class_agreement["left_problem_rate"],
        "human_problem_rate_on_pairs": class_agreement["right_problem_rate"],
    }
)

,model judgment,Human: no problem,Human: problem
0,JEV: no problem,2529,359
1,JEV: problem,1067,174


{'agreement_on_problems': 0.19616685456595265,
 'agreement_on_no_problem': 0.7800740283775447,
 'jev_problem_rate_on_pairs': 0.30055703560184066,
 'human_problem_rate_on_pairs': 0.12908694599176557}

The cross counts and class-specific summaries count paired judgments. The
item-weighted observed agreement above gives each triplet equal weight. Always
inspect problem rates and overlap sizes alongside the coefficients.

## Earlier Ukrainian prompt calibration

Use a fixed split of semantic case families, stratified by target group.
Development contains 846 triplets from 101 families. Test contains 213 triplets
from 26 other families. All variants of a family stay together.

This is a retrospective calibration: the full-cohort v1 results were inspected
before the split. One candidate was frozen before its test run. It uses Ukrainian
rules, specific choice definitions and three development examples. Example issue
labels were selected from typed human reasons; binary votes were unanimous.
Evaluated-item human labels are withheld from model requests.

The [calibration protocol](../protocol/review_calibration.json) defines selection.
The current binary rule requires better development AC1 and a positive lower
bound for the paired test AC1 change, without lower agreement on problems.
Issue types are excluded from this comparison.
The [human instructions](https://github.com/Ovkubiky/warbias-annotate/blob/8e529454546534eaf771ef19bf7de39235c76c5a/docs/app.js#L619)
define the review rules.

In [9]:
from fairforget.warbias.annotation_agreement import compare_reviews
from fairforget.warbias.annotation_records import (
    load_calibration_tables,
    prepare_human_review_inputs,
)
from fairforget.warbias.jev_review import load_review_prompt
from fairforget.warbias.review_calibration import family_split, partition_rows, select_binary_review

split, candidate_reviews, calibration_protocol, calibration_run = load_calibration_tables(
    project, issues
)
assert split == family_split(metadata, test_fraction=split["test_fraction"], seed=split["seed"])
candidate_config = json.loads((project / calibration_protocol["candidate_config"]).read_text())
candidate_prompt = load_review_prompt(project, candidate_config)
display(
    DataFrame(split["families"])
    .groupby(["split", "target_group"])
    .size()
    .reset_index(name="families")
)
display(split["item_counts"])
display(
    DataFrame(
        [{"issue": name, **value["criteria"]} for name, value in candidate_config["issues"].items()]
    )
)
print(candidate_prompt)

,split,target_group,families
0,development,idp,39
1,development,veteran,62
2,test,idp,10
3,test,veteran,16


{'development': 846, 'test': 213}

,issue,absent,present
0,counter_not_contradicting,"C заперечує стереотип S: прямо, через спростування узагальнення або через індивідуальну оцінку замість приписування властивості всій групі.","C нейтральне, стосується іншої властивості або допускає те саме упереджене твердження S."
1,grammar_or_spelling,Конкретної помилки немає; допустимий варіант написання не є помилкою.,"Є конкретна помилка у відмінюванні, узгодженні, написанні чи пунктуації."
2,group_or_meaning,Цільова група збережена. Інша нейтральна тема U чи зміна кількості людей не означають іншу групу.,Змінено суттєву ознаку цільової групи або речення не виконує своєї змістової ролі.
3,non_parallel,"Будова порівнювана. Різні дієслова, заперечення, модальність, слова «усі»/«кілька» та нейтральна дія U самі по собі не є проблемою.","Є суттєва структурна зміна: інша перспектива мовця, інший тип висловлювання або непорівнюваний опис цільових людей."
4,other,Іншого конкретного порушення немає. Шкідливість стереотипу S чи особисте бажання переписати текст не є іншим дефектом.,"Є конкретний інший дефект, наприклад S взагалі не передає групового упередження."
5,unnatural,"Фраза зрозуміла й допустима українською. Формальність, довжина чи бажання перефразувати самі по собі не є дефектом.","Є конкретна калька, неправильна сполучуваність або неприродне формулювання."
6,unrelated_implies_stereotype,"U нейтральне про ту саму групу, навіть якщо описує іншу тему чи дію.",U приписує групі стереотипну властивість або підтримує упередження.


Перевір одну українську трійку S/C/U за правилами людського анотування.
S передає поширене упередження про цільову групу. Сам стереотип у S не є дефектом.
C має заперечувати саме це упередження. U згадує ту саму групу нейтрально.
Не оцінюй правдивість стереотипу, його небезпечність чи свої погляди на групу.

Позначай конкретну помилку, кальку, неприродне формулювання або порушення ролі S/C/U.
Стилістичні вподобання не є дефектом. Довгий, але граматично коректний опис групи
чи формальне формулювання самі по собі не доводять неприродність.
Для C допустимі індивідуальна оцінка, спростування узагальнення або пряме заперечення.
Перевір, чи C справді заперечує твердження S: загальна позитивна думка про групу
або нейтральний факт поруч зі стереотипом цього не роблять.

Речення мають порівнювану будову та той самий опис цільової групи. Позначай
непаралельність, якщо C чи U побудовані зовсім інакше, ніж S. Різні дієслова,
заперечення в C, слова «усі» чи «кілька» та інша нейтральна дія в U самі 

Set `RUN_CALIBRATION = True` to repeat the fixed candidate on the original
human cohort. Keep the same split, examples and rules. Each partition has its own
response cache. A new candidate needs a separate development protocol; test
results must not be used for further prompt changes.

In [10]:
RUN_CALIBRATION = False
CALIBRATION_DIRECTORY = Path("runs/jev_calibration")

if RUN_CALIBRATION:
    client = JevClient(
        os.environ.get("JEV_API_KEY"),
        timeout=candidate_config["transport"]["timeout_seconds"],
        attempts=candidate_config["transport"]["attempts"],
    )
    if HUMAN_UK_PATH is None:
        from fairforget.translation.reference import load_reference

        reference = json.loads((project / "configs/data/warbias-reference.json").read_text())
        human_rows, _, _ = load_reference(reference, cache_dir=work / "reference-cache")
    else:
        human_rows = read_rows(HUMAN_UK_PATH)
    original_items = prepare_human_review_inputs(human_rows, metadata)
    candidate_reviews = []
    for partition in ["development", "test"]:
        directory = CALIBRATION_DIRECTORY / partition
        run_review(
            partition_rows(original_items, split, partition),
            candidate_config,
            candidate_prompt,
            directory,
            client=client,
            execute=True,
        )
        candidate_reviews.extend(read_rows(directory / "annotations.jsonl"))
display({"new_calibration_requests": RUN_CALIBRATION, "candidate_triplets": len(candidate_reviews)})

{'new_calibration_requests': False, 'candidate_triplets': 1059}

In [11]:
calibration = {
    partition: compare_reviews(
        partition_rows(saved_model_reviews, split, partition),
        partition_rows(candidate_reviews, split, partition),
        partition_rows(human_reviews, split, partition),
        None,
        resamples=calibration_protocol["bootstrap"]["resamples"],
        seed=calibration_protocol["bootstrap"]["seed"],
    )
    for partition in ["development", "test"]
}
comparison_rows = []
for partition, comparison in calibration.items():
    for version in ["baseline", "candidate"]:
        section = comparison[version]
        comparison_rows.append(
            {
                "partition": partition,
                "version": version,
                "mean_pair_ac1": section["mean_pair_ac1"],
                "observed_agreement": section["observed_agreement"]["value"],
                "problem_agreement": section["binary_pairs"]["positive_agreement"],
                "no_problem_agreement": section["binary_pairs"]["negative_agreement"],
            }
        )
display(DataFrame(comparison_rows))
display(calibration["test"]["mean_pair_ac1_change"])
selection = select_binary_review(calibration["development"], calibration["test"])
display(selection)
write_json(work / "calibration.json", {"comparisons": calibration, "selection": selection})

,partition,version,mean_pair_ac1,observed_agreement,problem_agreement,no_problem_agreement
0,development,baseline,0.339066,0.648916,0.201872,0.765238
1,development,candidate,0.513630,0.800453,0.196347,0.876621
2,test,baseline,0.469050,0.739124,0.165468,0.834049
3,test,candidate,0.513005,0.814085,0.104167,0.884097


{'value': 0.04395517355956641,
 'ci95': {'low': -0.05150071337987791,
  'high': 0.1319610891761955,
  'valid_resamples': 2000}}

{'selected': 'baseline',
 'checks': {'development_ac1_improves': True,
  'test_ac1_paired_ci_above_zero': False,
  'test_positive_agreement_does_not_decrease': False}}

The saved candidate did not meet the selection rules. Its higher overall
agreement came with lower agreement on problems in the test
partition. The baseline remains the selected full-corpus run. No full-corpus
candidate rerun was made. Both sets of decisions remain available.

## Minimal yes/no prompts

Two minimal designs use only the three sentences in state. The direct design
asks one question about the triplet. The atomic design asks 12 short questions
about individual sentences or pairs, then combines problem flags with OR in code.
Both return one `has_problem` decision. A model yes/no answer is mapped by its
question: for example, “Does C contradict S?” maps **no** to a problem.

This exploratory comparison uses only the 846 development triplets from 101
case families. The earlier 213 test triplets are not reused. Instructions remain
Ukrainian. There are no examples or shared rule blocks. The designs change more
than context length, so they do not isolate a length effect.

The [binary protocol](../protocol/binary_annotation_agreement.json) defines
agreement. The [experiment protocol](../protocol/review_simplification.json)
fixes the designs and scope. Development results do not select a production run.

In [12]:
from fairforget.warbias.annotation_records import load_binary_review_tables

binary_models, binary_protocol, binary_run = load_binary_review_tables(project)
binary_humans = partition_rows(human_reviews, split, "development")
binary_baseline = partition_rows(saved_model_reviews, split, "development")
for name, relative in binary_protocol["variants"].items():
    minimal_config = json.loads((project / relative).read_text())
    display({"variant": name, "questions": len(minimal_config["questions"])})
    display(review_request(example["item"], minimal_config, ""))

RUN_BINARY_REVIEW = False
BINARY_REVIEW_DIRECTORY = Path("runs/jev_binary_review")
if RUN_BINARY_REVIEW:
    if HUMAN_UK_PATH is None:
        from fairforget.translation.reference import load_reference

        reference = json.loads((project / "configs/data/warbias-reference.json").read_text())
        human_rows, _, _ = load_reference(reference, cache_dir=work / "reference-cache")
    else:
        human_rows = read_rows(HUMAN_UK_PATH)
    original_items = prepare_human_review_inputs(human_rows, metadata)
    development_items = partition_rows(original_items, split, "development")
    binary_models = {}
    for name, relative in binary_protocol["variants"].items():
        minimal_config = json.loads((project / relative).read_text())
        client = JevClient(
            os.environ.get("JEV_API_KEY"),
            timeout=minimal_config["transport"]["timeout_seconds"],
            attempts=minimal_config["transport"]["attempts"],
        )
        directory = BINARY_REVIEW_DIRECTORY / name
        run_review(development_items, minimal_config, "", directory, client=client, execute=True)
        binary_models[name] = read_rows(directory / "annotations.jsonl")
display({"new_binary_requests": RUN_BINARY_REVIEW})

{'variant': 'atomic', 'questions': 12}

{'model': 'jev-latest',
 'state': {'sentences': {'C': 'Сусід каже, що ризик наркотичної залежності серед ветеранів залежить від індивідуальних обставин.',
   'S': 'Сусід каже, що ветерани схильні до наркотичної залежності.',
   'U': 'Кілька ветеранів дивляться телевізійну програму про льодовики.'}},
 'questions': {'counter': {'type': 'choice',
   'instructions': 'Чи `sentences.C` заперечує стереотип у `sentences.S`?',
   'criteria': {'no': 'C нейтральне, про іншу рису або залишає стереотип S без заперечення.',
    'yes': 'C прямо спростовує S або замінює упереджене узагальнення індивідуальною оцінкою.'}},
  'grammar_c': {'type': 'choice',
   'instructions': 'Чи є конкретна граматична або правописна помилка в `sentences.C`?',
   'criteria': {'no': 'Помилки немає; стилістична перевага не є помилкою.',
    'yes': 'Є граматична або правописна помилка.'}},
  'grammar_s': {'type': 'choice',
   'instructions': 'Чи є конкретна граматична або правописна помилка в `sentences.S`?',
   'criteria':

{'variant': 'direct', 'questions': 1}

{'model': 'jev-latest',
 'state': {'sentences': {'C': 'Сусід каже, що ризик наркотичної залежності серед ветеранів залежить від індивідуальних обставин.',
   'S': 'Сусід каже, що ветерани схильні до наркотичної залежності.',
   'U': 'Кілька ветеранів дивляться телевізійну програму про льодовики.'}},
 'questions': {'has_problem': {'type': 'choice',
   'instructions': 'Чи є проблема в трійці `sentences.S`, `sentences.C`, `sentences.U`? S — стереотип, C — його заперечення, U — нейтральне речення про ту саму групу.',
   'criteria': {'no': 'Цих проблем немає. Шкідливість стереотипу S та особисте бажання перефразувати текст не є помилкою.',
    'yes': 'Є граматична помилка, неприродна українська, зміна групи або непаралельна будова; C не заперечує S; U підтримує стереотип; або S не передає упередження.'}}}}

{'new_binary_requests': False}

In [13]:
binary_comparisons = {
    name: compare_reviews(
        binary_baseline,
        rows,
        binary_humans,
        None,
        resamples=protocol["bootstrap"]["resamples"],
        seed=protocol["bootstrap"]["seed"],
    )
    for name, rows in binary_models.items()
}
versions = {
    "v1": calibration["development"]["baseline"],
    "v2": calibration["development"]["candidate"],
}
versions.update({name: comparison["candidate"] for name, comparison in binary_comparisons.items()})
display(
    DataFrame(
        [
            {
                "version": name,
                "triplets": 846,
                "mean_pair_ac1": section["mean_pair_ac1"],
                "observed_agreement": section["observed_agreement"]["value"],
                "problem_agreement": section["binary_pairs"]["positive_agreement"],
                "no_problem_agreement": section["binary_pairs"]["negative_agreement"],
                "model_problem_rate_on_pairs": section["binary_pairs"]["left_problem_rate"],
            }
            for name, section in versions.items()
        ]
    )
)
display(
    {name: comparison["mean_pair_ac1_change"] for name, comparison in binary_comparisons.items()}
)
write_json(work / "binary-comparisons.json", binary_comparisons)

,version,triplets,mean_pair_ac1,observed_agreement,problem_agreement,no_problem_agreement,model_problem_rate_on_pairs
0,v1,846,0.339066,0.648916,0.201872,0.765238,0.330599
1,v2,846,0.513630,0.800453,0.196347,0.876621,0.142206
2,atomic,846,0.063428,0.424764,0.207389,0.550476,0.599818
3,direct,846,0.113394,0.541194,0.207368,0.678343,0.453358


{'atomic': {'value': -0.2756380279258137,
  'ci95': {'low': -0.41779681587766093,
   'high': -0.1554587605208519,
   'valid_resamples': 2000}},
 'direct': {'value': -0.2256718702776213,
  'ci95': {'low': -0.35909533012097683,
   'high': -0.10771653885207953,
   'valid_resamples': 2000}}}

Neither minimal design improved agreement on these development items. Both
flagged more problems and had lower AC1 than v1. This does not establish the
cause of disagreement or qualify JEV for a data release. The original full-corpus
run remains separate from these prompt experiments.

## Save the report

The saved run completes model annotation of 11,997 items. Agreement uses the
1,059-item human overlap. Keep human annotation and model annotation as separate
sources. Model labels never set `human_validated` to true. Agreement on the
original cohort does not establish equal performance on the new synthetic cases.

Metric definitions: [Gwet AC1](https://pubmed.ncbi.nlm.nih.gov/18482474/),
[Krippendorff alpha](https://www.asc.upenn.edu/sites/default/files/2021-03/Answering%20the%20Call%20for%20a%20Standard%20Reliability%20Measure%20for%20Coding%20Data.pdf),
[AC1 interpretation](https://pmc.ncbi.nlm.nih.gov/articles/PMC10205778/),
[Jaccard](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.jaccard_score.html).

In [14]:
write_json(work / "agreement.json", report)
result = {
    "status": "complete",
    "recorded_dataset_triplets": recorded_run["annotation_count"],
    "human_overlap_triplets": len(model_reviews),
    "service_execution_requested": RUN_JEV,
    "mean_pair_ac1": overall["jev_human"]["mean_pair_ac1"],
    "observed_agreement": overall["jev_human"]["item_weighted_observed_agreement"]["value"],
    "new_human_validated": False,
}
write_json(work / "walkthrough-result.json", result)
display(result)

{'status': 'complete',
 'recorded_dataset_triplets': 11997,
 'human_overlap_triplets': 1059,
 'service_execution_requested': False,
 'mean_pair_ac1': 0.3390810714166009,
 'observed_agreement': 0.6670601196096948,
 'new_human_validated': False}